# Adding a Custom Generator — LIHEAP Example

This notebook is a step-by-step guide for contributors who want to add support
for a new benefits program. It uses **LIHEAP** (Low Income Home Energy Assistance
Program) as the example — but the same pattern applies to any new program.

By the end you will have:
- A minimal `LIHEAPSource` that returns income thresholds
- A `LIHEAPEligibilityGenerator` that produces valid `TestCase` objects
- A `RationaleTrace` grounded in real LIHEAP policy
- Verified output that passes `TestCase.is_valid()`

To graduate a generator from notebook to production, see **CLAUDE.md §
"Adding a New Program"**.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
print('govsynth path inserted')

## 1. Understanding the Base Classes

A generator does not extend a formal `Generator` ABC — it is a plain class
with a `generate(n, profile_strategy, seed)` method that returns `list[TestCase]`.

The key data structures are:

| Class | Purpose |
|---|---|
| `TestCase` | The primary output unit — scenario + task + answer + rationale |
| `ScenarioBlock` | The household situation presented to the model |
| `TaskBlock` | The instruction given to the model |
| `RationaleTrace` | Ordered list of `ReasoningStep` objects + conclusion |
| `ReasoningStep` | One rule applied, one computation, one result |
| `PolicyCitation` | A CFR or handbook citation grounding the trace |

In [ ]:
from govsynth.models.test_case import TestCase, ScenarioBlock, TaskBlock
from govsynth.models.rationale import RationaleTrace, ReasoningStep, PolicyCitation
from govsynth.models.enums import Difficulty, Program, TaskType

# Inspect the TestCase required fields
import inspect
for name, field in TestCase.model_fields.items():
    req = 'required' if field.is_required() else f'default={field.default!r}'
    print(f'  {name:<25} {req}')

## 2. Step 1 — Define a Minimal DataSource

LIHEAP income eligibility is typically set at **150% FPL** (some states use
60% of State Median Income, but 150% FPL is the federal floor).

We define a minimal source that returns hardcoded thresholds. In production
this would load from `data/thresholds/liheap_2025.json`.

In [ ]:
from dataclasses import dataclass

# 2025 HHS Poverty Guidelines (48 contiguous states + DC)
# Source: HHS, 90 Fed. Reg. 3121 (Jan. 15, 2025)
_FPL_2025 = {1: 15650, 2: 21150, 3: 26650, 4: 32150, 5: 37650, 6: 43150, 7: 48650, 8: 54150}
_FPL_EACH_ADDITIONAL = 5500  # for HH > 8
_LIHEAP_PCT_FPL = 1.50  # 150% FPL — federal maximum


@dataclass
class LIHEAPLimits:
    household_size: int
    annual_gross_limit: float
    monthly_gross_limit: float


class LIHEAPSource:
    """Minimal LIHEAP income threshold source (illustrative, not production-ready)."""

    def __init__(self, fiscal_year: int = 2025, state: str = "VA") -> None:
        self.fiscal_year = fiscal_year
        self.state = state.upper()

    def by_household_size(self, size: int) -> LIHEAPLimits:
        """Return the 150% FPL monthly income limit for a given household size."""
        if size <= 8:
            annual_fpl = _FPL_2025[size]
        else:
            annual_fpl = _FPL_2025[8] + (size - 8) * _FPL_EACH_ADDITIONAL
        annual_limit = round(annual_fpl * _LIHEAP_PCT_FPL, 2)
        return LIHEAPLimits(
            household_size=size,
            annual_gross_limit=annual_limit,
            monthly_gross_limit=round(annual_limit / 12, 2),
        )

    def is_income_eligible(self, household_size: int, monthly_gross_income: float) -> tuple[bool, str]:
        limits = self.by_household_size(household_size)
        if monthly_gross_income <= limits.monthly_gross_limit:
            return True, f"Eligible: ${monthly_gross_income:,.2f} ≤ ${limits.monthly_gross_limit:,.2f} (150% FPL)"
        return False, f"Ineligible: ${monthly_gross_income:,.2f} > ${limits.monthly_gross_limit:,.2f} (150% FPL)"


# Verify the thresholds look right
src = LIHEAPSource(fiscal_year=2025, state="VA")
print(f'{"HH":<5} {"150% FPL (monthly)"}')
print('-' * 30)
for sz in range(1, 7):
    lim = src.by_household_size(sz)
    print(f'{sz:<5} ${lim.monthly_gross_limit:,.2f}')

## 3. Step 2 — Build a RationaleTrace

A `RationaleTrace` must have **at least 2 `ReasoningStep` objects** — this is
enforced by `TestCase` model validation. Each step needs:
- `step_number`: 1-based index
- `title`: short label
- `rule_applied`: CFR section or policy citation
- `computation`: plain language description of the check
- `result`: PASS / FAIL / computed value

In [ ]:
def build_liheap_trace(
    household_size: int,
    monthly_gross_income: float,
    limit: LIHEAPLimits,
    eligible: bool,
) -> RationaleTrace:
    """Build a RationaleTrace for a LIHEAP income eligibility determination."""
    step1 = ReasoningStep(
        step_number=1,
        title="Identify applicable income limit",
        rule_applied="42 U.S.C. § 8624(b)(2); 45 CFR § 96.85",
        inputs={"household_size": household_size, "pct_fpl": "150%"},
        computation=(
            f"150% FPL for {household_size}-person household: "
            f"${limit.monthly_gross_limit:,.2f}/month"
        ),
        result=f"Limit = ${limit.monthly_gross_limit:,.2f}/month",
    )
    step2 = ReasoningStep(
        step_number=2,
        title="Compare gross income to limit",
        rule_applied="42 U.S.C. § 8624(b)(2)",
        inputs={"monthly_gross_income": monthly_gross_income, "limit": limit.monthly_gross_limit},
        computation=(
            f"${monthly_gross_income:,.2f} {'≤' if eligible else '>'} "
            f"${limit.monthly_gross_limit:,.2f}"
        ),
        result="PASS" if eligible else "FAIL",
        is_determinative=True,
    )
    outcome_label = "ELIGIBLE" if eligible else "INELIGIBLE"
    return RationaleTrace(
        steps=[step1, step2],
        conclusion=(
            f"{outcome_label}. Gross income ${monthly_gross_income:,.2f}/month "
            f"{'does not exceed' if eligible else 'exceeds'} the 150% FPL limit of "
            f"${limit.monthly_gross_limit:,.2f}/month for a {household_size}-person household."
        ),
        policy_basis=[
            PolicyCitation(
                document="Low Income Home Energy Assistance Act",
                section="42 U.S.C. § 8624(b)(2)",
                year=2025,
            )
        ],
    )


# Test it
lim = src.by_household_size(3)
trace = build_liheap_trace(3, 2800.0, lim, eligible=True)
print(trace.to_plain_text())

## 4. Step 3 — Implement the Generator

The generator is a plain class with a `generate()` method. It uses
`USHouseholdProfile.at_threshold` for edge cases and `USHouseholdProfile.random`
for other strategies.

Note the **case ID format**: `liheap.va.eligibility.income_limit.<descriptor>.<hex>`

In [ ]:
import random
import uuid
from govsynth.profiles.us_household import USHouseholdProfile

_LIHEAP_TASK = (
    "Based on the household's situation described above, determine whether this household "
    "is eligible for LIHEAP (Low Income Home Energy Assistance Program) benefits. "
    "Show your reasoning step by step, citing the specific regulations that apply. "
    "State your final determination (eligible or ineligible)."
)

_OFFSETS = [0.0, 0.01, -0.01, 0.05, -0.05]


class LIHEAPEligibilityGenerator:
    """Generates LIHEAP income eligibility test cases (illustrative).

    Args:
        fiscal_year: Program year for thresholds.
        state: Two-letter state code.
    """

    def __init__(self, fiscal_year: int = 2025, state: str = "VA") -> None:
        self.fiscal_year = fiscal_year
        self.state = state.upper()
        self.source = LIHEAPSource(fiscal_year=fiscal_year, state=state)

    def generate(
        self,
        n: int,
        profile_strategy: str = "edge_saturated",
        seed: int | None = None,
    ) -> list[TestCase]:
        """Generate n LIHEAP eligibility test cases."""
        rng = random.Random(seed)
        cases: list[TestCase] = []

        hh_sizes = [1, 2, 3, 4]
        offset_cycle = _OFFSETS * (n // len(_OFFSETS) + 1)

        for i in range(n):
            case_seed = rng.randint(0, 2**31)
            hh_size = hh_sizes[i % len(hh_sizes)]
            offset = offset_cycle[i]
            limit = self.source.by_household_size(hh_size)

            if profile_strategy in ("edge_saturated", "uniform"):
                gross = round(limit.monthly_gross_limit * (1 + offset), 2)
                gross = max(gross, 100.0)
            else:
                profile = USHouseholdProfile.random(
                    state=self.state, seed=case_seed, strategy=profile_strategy
                )
                gross = profile.monthly_gross_income
                hh_size = profile.household_size
                limit = self.source.by_household_size(hh_size)

            eligible, reason = self.source.is_income_eligible(hh_size, gross)
            trace = build_liheap_trace(hh_size, gross, limit, eligible)

            descriptor = "below_limit" if eligible else "above_limit"
            disambiguator = uuid.UUID(int=case_seed).hex[:6]
            case_id = f"liheap.{self.state.lower()}.eligibility.income_limit.{descriptor}.hh{hh_size}.{disambiguator}"

            difficulty = Difficulty.HARD if abs(offset) <= 0.01 else Difficulty.MEDIUM

            profile_obj = USHouseholdProfile(
                household_size=hh_size,
                monthly_gross_income=gross,
                state=self.state,
            )

            scenario = ScenarioBlock(
                summary=profile_obj.natural_language_summary(program="liheap"),
                household_size=hh_size,
                monthly_gross_income=gross,
                state=self.state,
            )
            task = TaskBlock(instruction=_LIHEAP_TASK)

            outcome = "eligible" if eligible else "ineligible"
            answer = (
                f"This household is {outcome.upper()} for LIHEAP benefits ({self.fiscal_year}). "
                f"{reason}"
            )

            cases.append(
                TestCase(
                    case_id=case_id,
                    program="liheap",
                    jurisdiction=f"us.{self.state.lower()}",
                    task_type=TaskType.ELIGIBILITY,
                    difficulty=difficulty,
                    scenario=scenario,
                    task=task,
                    expected_outcome=outcome,
                    expected_answer=answer,
                    rationale_trace=trace,
                    variation_tags=["income_limit_150pct_fpl", descriptor, f"hh{hh_size}"],
                    source_citations=["42 U.S.C. § 8624(b)(2)"],
                    seed=case_seed,
                )
            )

        return cases


print('LIHEAPEligibilityGenerator defined.')

## 5. Step 4 — Generate and Inspect Cases

In [ ]:
generator = LIHEAPEligibilityGenerator(fiscal_year=2025, state="VA")
cases = generator.generate(n=8, seed=42)

print(f'Generated {len(cases)} cases')
print(f'Eligible:   {sum(1 for c in cases if c.expected_outcome == "eligible")}')
print(f'Ineligible: {sum(1 for c in cases if c.expected_outcome == "ineligible")}')
print()
print('Case IDs:')
for c in cases:
    print(f'  {c.case_id}')

In [ ]:
# Inspect the first case in full
case = cases[0]
print('=== CASE ID ===')
print(case.case_id)
print()
print('=== SCENARIO ===')
print(case.scenario.summary)
print(f'  HH size: {case.scenario.household_size}, Gross: ${case.scenario.monthly_gross_income:,.2f}')
print()
print('=== EXPECTED OUTCOME ===')
print(case.expected_outcome.upper())
print()
print('=== EXPECTED ANSWER ===')
print(case.expected_answer)
print()
print('=== RATIONALE TRACE ===')
print(case.rationale_trace.to_plain_text())
print()
print('=== VARIATION TAGS ===')
print(case.variation_tags)

## 6. Step 5 — Validate Against the Schema

In [ ]:
# Run TestCase.validate() on every generated case
all_valid = True
for c in cases:
    errors = c.validate()
    if errors:
        print(f'INVALID {c.case_id}: {errors}')
        all_valid = False
    else:
        print(f'  OK  {c.case_id}')

print()
print(f'All valid: {all_valid}')

In [ ]:
# Export to YAML and validate via the govsynth CLI
import os
from govsynth.formatters.yaml_fmt import YAMLFormatter

os.makedirs('./liheap_output', exist_ok=True)
fmt = YAMLFormatter()
for c in cases:
    path = f'./liheap_output/{c.case_id}.yaml'
    with open(path, 'w') as f:
        f.write(fmt.format_one(c))

print(f'Wrote {len(cases)} YAML files to ./liheap_output/')
print()

# Validate with the CLI
first_file = f'./liheap_output/{cases[0].case_id}.yaml'
!govsynth validate {first_file}

## 7. What's Next — Graduating to Production

To move this from a notebook prototype to a proper module in the library,
follow the checklist in **CLAUDE.md § "Adding a New Program"**:

1. **`data/thresholds/liheap_2025.json`** — replace hardcoded thresholds with
   a proper JSON file following the threshold schema. Cite the source regulation.

2. **`govsynth/sources/us/liheap.py`** — move `LIHEAPSource` here, extend
   `DataSource`, and load thresholds from the JSON file.

3. **`govsynth/generators/liheap_eligibility.py`** — move
   `LIHEAPEligibilityGenerator` here.

4. **`govsynth/presets.py`** — register `liheap.va` (and other states) in
   `PRESETS`.

5. **`tests/unit/test_liheap_source.py`** — unit tests for threshold loading
   and `is_income_eligible()`.

6. **`tests/unit/test_liheap_generator.py`** — verify case IDs, rationale
   structure, edge cases at the 150% FPL boundary.

7. **`docs/programs/liheap.md`** — policy background, threshold source,
   known state variations (some states use 60% SMI instead of 150% FPL).

After that, `govsynth generate liheap.va --n 50 --seed 42` will work out of
the box.